# Extract Patterns

Bazaar has three data sources.

An **orders database** (SQLite). **Daily CSV exports** from sellers. An **API** that returns product catalogue updates.

Your pipeline needs to pull from all three. That is the "E" in ETL: *extraction*. It sounds simple enough -- read data from a source, hand it to the next stage. But the details trip people up. Different formats, different access patterns, and one absolutely critical question: *how do you avoid pulling the same data twice?*

We will work through each source, see the naive approach fail, and then fix it properly.

## Setup

We need `sqlalchemy` for database access. Pandas uses it under the hood when you call `read_sql`.

In [ ]:
%pip install -q sqlalchemy

In [ ]:
import pandas as pd
import json
import glob
import os
from sqlalchemy import create_engine

## Source 1: The Orders Database (SQLite)

Bazaar stores orders in a SQLite database. It has three tables: `orders`, `customers`, and `sellers`.

SQLite is just a file on disk -- no server needed. That makes it perfect for development and testing. In production you would likely connect to PostgreSQL or similar, but the extraction pattern is identical.

Let's connect and see what we are working with.

In [ ]:
engine = create_engine("sqlite:///../data/bazaar_orders.sqlite")

# Quick look at the tables
with engine.connect() as conn:
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
    print(tables)

The simplest extraction is a `SELECT *`. Pull everything.

In [ ]:
with engine.connect() as conn:
    orders_df = pd.read_sql("SELECT * FROM orders", conn)

print(f"Extracted {len(orders_df)} orders")
orders_df.head()

In [ ]:
orders_df.dtypes

That works. We have all 2,000 orders in a DataFrame. Notice that `order_date` came back as a string -- SQLite does not have a native datetime type, so pandas treats it as text. We will deal with that in the transform stage.

Let's also grab the customers and sellers tables while we are here.

In [ ]:
with engine.connect() as conn:
    customers_df = pd.read_sql("SELECT * FROM customers", conn)
    sellers_df = pd.read_sql("SELECT * FROM sellers", conn)

print(f"Customers: {len(customers_df)}, Sellers: {len(sellers_df)}")
sellers_df.head()

### Filtering at source

Pulling everything works for small datasets. But if the orders table had 200 million rows, you would not want to load them all into memory. Push the filtering down to SQL.

In [ ]:
# Only completed orders from March 2024
query = """
    SELECT *
    FROM orders
    WHERE status = 'completed'
      AND order_date >= '2024-03-01'
    ORDER BY order_id
"""

with engine.connect() as conn:
    march_orders = pd.read_sql(query, conn)

print(f"Completed March orders: {len(march_orders)}")
march_orders.head()

Good. Filtering at source means less data over the wire, less memory used, faster extraction. Always prefer this over extracting everything and filtering in Python.

## Source 2: Daily Seller CSV Files

Bazaar's sellers upload CSV exports every day. Each file follows the naming pattern `seller_{name}_{date}.csv`. Three sellers -- Alpha, Beta, and Gamma -- each with files on different dates.

We need to discover these files, read them all, and combine them into a single DataFrame. The `glob` module is our friend here.

In [ ]:
# Find all seller CSV files
csv_files = sorted(glob.glob("../data/daily_seller_files/*.csv"))
print(f"Found {len(csv_files)} files:")
for f in csv_files:
    print(f"  {os.path.basename(f)}")

We can filter by seller name using a more specific glob pattern.

In [ ]:
# Only Alpha's files
alpha_files = sorted(glob.glob("../data/daily_seller_files/seller_alpha_*.csv"))
print(f"Alpha has {len(alpha_files)} files")

# Only files from March 1st (any seller)
march1_files = sorted(glob.glob("../data/daily_seller_files/*_20240301.csv"))
print(f"March 1st files: {len(march1_files)}")

Now the standard pattern: read each CSV, tag it with the filename (so we know where it came from), and concatenate them.

In [ ]:
frames = []

for filepath in csv_files:
    df = pd.read_csv(filepath)
    # Extract seller name and date from the filename
    basename = os.path.basename(filepath)          # seller_alpha_20240301.csv
    parts = basename.replace(".csv", "").split("_")  # ['seller', 'alpha', '20240301']
    df["seller_name"] = parts[1]
    df["file_date"] = parts[2]
    df["source_file"] = basename
    frames.append(df)

all_sellers = pd.concat(frames, ignore_index=True)
print(f"Combined: {len(all_sellers)} rows from {len(csv_files)} files")
all_sellers.head()

In [ ]:
# Check the breakdown by seller
all_sellers.groupby("seller_name").size()

Notice that some products appear in multiple files -- a product listed on March 1st might also appear on March 2nd with updated stock. That is expected. We will handle deduplication in the transform stage.

For now, the extractor's job is to faithfully pull all the data and tag it with its source.

## Source 3: The Product Catalogue API (JSON)

Bazaar's product service exposes an API that returns catalogue updates as JSON. In production, you would use `requests` to call the API. Here, we have a JSON file that simulates the response.

The structure is typical of a paginated API: metadata at the top, data in a nested list.

In [ ]:
with open("../data/marketplace_api.json", "r") as f:
    api_response = json.load(f)

# Inspect the top-level keys
print("Top-level keys:", list(api_response.keys()))
print(f"API version: {api_response['api_version']}")
print(f"Timestamp: {api_response['timestamp']}")
print(f"Total results: {api_response['total_results']}")

In [ ]:
# The products are nested under the 'products' key
products_df = pd.DataFrame(api_response["products"])
print(f"Extracted {len(products_df)} products from API")
products_df.head()

With real APIs, you often need to handle pagination:

```python
all_products = []
page = 1

while True:
    response = requests.get(f"{base_url}/products?page={page}").json()
    all_products.extend(response["products"])
    if len(response["products"]) < response["per_page"]:
        break
    page += 1

df = pd.DataFrame(all_products)
```

Our simulated API returns everything in one page, but the extraction pattern is the same: read the response, pull out the data, put it in a DataFrame.

## The Duplicate Problem

We have three working extractors. But there is a problem. Watch what happens when we run the orders extraction twice.

In [ ]:
# Simulate running the extractor twice and appending the results
with engine.connect() as conn:
    batch_1 = pd.read_sql("SELECT * FROM orders", conn)

with engine.connect() as conn:
    batch_2 = pd.read_sql("SELECT * FROM orders", conn)

combined = pd.concat([batch_1, batch_2], ignore_index=True)
print(f"Batch 1: {len(batch_1)} rows")
print(f"Batch 2: {len(batch_2)} rows")
print(f"Combined: {len(combined)} rows")
print(f"Unique order IDs: {combined['order_id'].nunique()}")

4,000 rows but only 2,000 unique orders. Every single row is duplicated. If your pipeline runs nightly and you always extract everything, you get duplicates every single night.

"Just deduplicate later" is a common response. And yes, you can. But it wastes bandwidth, wastes memory, wastes time. With large datasets it becomes genuinely expensive. And if you are appending to a destination table, those duplicates pile up fast.

The proper solution is **incremental extraction**.

## The High-Water Mark

A high-water mark is a bookmark that records "how far we got" on the last extraction. Think of it like a bookmark in a book -- you do not re-read from the beginning every time you pick the book up.

For our orders table, the simplest bookmark is `last_extracted_order_id`. We track the highest order ID we have already extracted, and next time we only pull orders with a higher ID.

Here is the idea:

```
Run 1: Extract all orders (order_id 1 to 2000). Bookmark = 2000.
Run 2: Extract orders WHERE order_id > 2000. If nothing new, zero rows. No duplicates.
```

In [ ]:
def extract_orders(engine, last_order_id=0):
    """
    Extract orders from the database, starting after last_order_id.
    Returns (DataFrame, new_bookmark).
    """
    query = f"""
        SELECT *
        FROM orders
        WHERE order_id > {last_order_id}
        ORDER BY order_id
    """
    with engine.connect() as conn:
        df = pd.read_sql(query, conn)

    if len(df) > 0:
        new_bookmark = int(df["order_id"].max())
    else:
        new_bookmark = last_order_id

    return df, new_bookmark

In [ ]:
# First run: extract everything (bookmark starts at 0)
batch_1, bookmark = extract_orders(engine, last_order_id=0)
print(f"Run 1: extracted {len(batch_1)} rows, bookmark = {bookmark}")

In [ ]:
# Second run: use the bookmark
batch_2, bookmark = extract_orders(engine, last_order_id=bookmark)
print(f"Run 2: extracted {len(batch_2)} rows, bookmark = {bookmark}")

Zero rows on the second run. No duplicates. The bookmark works.

If new orders had been inserted between runs, only those new orders would be extracted. This is **idempotent extraction** -- running the extractor multiple times produces the correct result without side effects.

A few things to note:

- The bookmark column must be **monotonically increasing**. Auto-incrementing IDs and timestamps both work. Anything else is risky.
- You need to **persist the bookmark** somewhere between runs. In production, this is often a small metadata table or a state file.
- For timestamp-based bookmarks, be careful with clock skew and late-arriving data. IDs are generally safer.

### Date-based high-water mark for CSV files

The seller CSV files are named with dates. We can use the date in the filename as our bookmark: only read files newer than the last extraction date.

In [ ]:
def extract_seller_files(data_dir, last_extracted_date=None):
    """
    Extract seller CSV files, optionally filtering to files
    newer than last_extracted_date (format: YYYYMMDD).
    Returns (DataFrame, new_bookmark).
    """
    csv_files = sorted(glob.glob(os.path.join(data_dir, "*.csv")))
    frames = []

    for filepath in csv_files:
        basename = os.path.basename(filepath)
        parts = basename.replace(".csv", "").split("_")
        file_date = parts[2]  # e.g. '20240301'

        # Skip files we have already processed
        if last_extracted_date and file_date <= last_extracted_date:
            continue

        df = pd.read_csv(filepath)
        df["seller_name"] = parts[1]
        df["file_date"] = file_date
        frames.append(df)

    if frames:
        result = pd.concat(frames, ignore_index=True)
        new_bookmark = max(df["file_date"] for df in [pd.DataFrame({"file_date": [p.replace(".csv", "").split("_")[2]]}) for p in [os.path.basename(f) for f in csv_files if os.path.basename(f).replace(".csv", "").split("_")[2] > (last_extracted_date or "")]])
        # Simpler: just get the max file_date from what we read
        new_bookmark = result["file_date"].max()
    else:
        result = pd.DataFrame()
        new_bookmark = last_extracted_date

    return result, new_bookmark

In [ ]:
# First run: get everything
seller_data, bookmark = extract_seller_files("../data/daily_seller_files")
print(f"Run 1: {len(seller_data)} rows, bookmark = {bookmark}")

In [ ]:
# Second run: nothing new
seller_data_2, bookmark = extract_seller_files("../data/daily_seller_files", last_extracted_date=bookmark)
print(f"Run 2: {len(seller_data_2)} rows, bookmark = {bookmark}")

Same pattern, different implementation. The bookmark tracks which files we have already seen, and the extractor skips them on subsequent runs.

## Wrapping extractors in functions

You will have noticed that we have been writing functions for our extractors. This is deliberate. A good extractor:

1. Takes configuration as parameters (connection string, file path, bookmark)
2. Returns a DataFrame (and optionally an updated bookmark)
3. Does not mutate anything outside itself -- no side effects

This makes extractors **testable**, **composable**, and **reusable**. We will lean on this pattern heavily in notebook 4 when we build the full pipeline.

## Putting it together: a multi-source extractor

In [ ]:
def extract_api_products(filepath):
    """
    Extract products from the marketplace API JSON file.
    Returns a DataFrame.
    """
    with open(filepath, "r") as f:
        response = json.load(f)

    df = pd.DataFrame(response["products"])
    df["api_timestamp"] = response["timestamp"]
    return df

In [ ]:
# Extract from all three sources
orders, orders_bookmark = extract_orders(engine, last_order_id=0)
seller_products, seller_bookmark = extract_seller_files("../data/daily_seller_files")
api_products = extract_api_products("../data/marketplace_api.json")

print(f"Orders:          {len(orders):>5} rows")
print(f"Seller products: {len(seller_products):>5} rows")
print(f"API products:    {len(api_products):>5} rows")

Three sources, three extractors, three DataFrames. Each one is independent, testable, and bookmark-capable. This is the foundation of any ETL pipeline.

## Summary

What we covered:

- **SQLite extraction** with `pd.read_sql` and SQLAlchemy
- **File discovery** with `glob.glob` for batches of CSVs
- **JSON extraction** from simulated API responses
- **The duplicate problem** -- extracting without a bookmark causes duplicates
- **High-water marks** -- track the last extracted ID or date, only pull new records
- **Idempotent extraction** -- running the extractor twice produces the correct result
- **Wrapping extractors in functions** -- parameters in, DataFrame out, no side effects

---

## Exercises

### Exercise 1: Filtered extraction

Write a function `extract_orders_by_status` that takes the engine and a `status` string (e.g. `"completed"`, `"refunded"`) and returns only orders with that status. It should also accept an optional `last_order_id` parameter for incremental extraction.

In [ ]:
# Your code here


### Exercise 2: Single-seller extraction

Write a function `extract_seller` that takes a `seller_name` (e.g. `"alpha"`) and a `data_dir` path, and returns a DataFrame of all CSV files for that seller only. Include the `file_date` column.

In [ ]:
# Your code here


### Exercise 3: Active products only

The API response includes an `active` field. Write a function `extract_active_products` that reads the API JSON and returns only products where `active` is `True`.

In [ ]:
# Your code here


### Exercise 4: Timestamp-based bookmark

The orders table has an `order_date` column. Write an alternative version of `extract_orders` that uses `order_date` as the high-water mark instead of `order_id`. What are the trade-offs compared to using the ID?

In [ ]:
# Your code here
